# LAB-HW-02 — KV260 供电与 target discovery

**今天只解决一个问题：正确给 KV260 供电，并让 development host 通过 JTAG 发现真实器件。**

今天仍然不写 neuron RTL、不生成 bitstream、不学习 AXI。

**前置：LAB-HW-00、LAB-HW-01。**  
**Project Trace:** RMD-012 · T-HW-002/T-HW-011

## 1. 今天桌上应该有什么

- 已完成 LAB-HW-00 的 development host；
- 已完成 inventory 的 KV260；
- **12 V / 3 A**、center-positive、符合 KV260 规格的电源；
- 能传数据的 USB cable，连接 development host 与 **J4**；
- 今天不需要 microSD 才能完成 JTAG target discovery。

AMD DS986 给出的推荐 DC input 是 +12 V / 3 A。不要把“USB 已插入”当成主板已经供电。

## 2. Preflight

开始前逐项确认：

- [ ] KV260 当前未通 12 V；
- [ ] USB cable 是 data cable，不是 charge-only；
- [ ] USB 连接的是 **J4 FTDI UART/JTAG**；
- [ ] 12 V 电源将连接 **J12**；
- [ ] LAB-HW-00 preflight 已通过；
- [ ] 没有向 Pmod/GPIO 接任何未知电压。

## 3. Connection Map

<svg xmlns="http://www.w3.org/2000/svg" width="760" height="330" viewBox="0 0 760 330" role="img" aria-label="KV260 LAB-HW-02 power and JTAG connection map">
  <rect x="30" y="115" width="170" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="115" y="145" text-anchor="middle" font-size="16">development host</text>
  <text x="115" y="170" text-anchor="middle" font-size="13">Vivado 2026.1</text>

  <rect x="295" y="55" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="385" y="85" text-anchor="middle" font-size="16">J4 FTDI</text>
  <text x="385" y="110" text-anchor="middle" font-size="13">USB data / JTAG / UART</text>

  <rect x="560" y="55" width="160" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="640" y="85" text-anchor="middle" font-size="16">K26 target</text>
  <text x="640" y="110" text-anchor="middle" font-size="13">JTAG device</text>

  <rect x="30" y="245" width="170" height="60" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="115" y="280" text-anchor="middle" font-size="15">12 V / 3 A supply</text>

  <rect x="295" y="235" width="180" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="385" y="265" text-anchor="middle" font-size="16">J12</text>
  <text x="385" y="288" text-anchor="middle" font-size="13">main board power</text>

  <path d="M200 140 L295 105" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="295,105 285,101 288,112" fill="#333"/>
  <text x="245" y="110" text-anchor="middle" font-size="12">USB data</text>

  <path d="M475 95 L560 95" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="560,95 550,90 550,100" fill="#333"/>
  <text x="518" y="82" text-anchor="middle" font-size="12">JTAG</text>

  <path d="M200 275 L295 275" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="295,275 285,270 285,280" fill="#333"/>
  <text x="247" y="261" text-anchor="middle" font-size="12">12 V DC</text>

  <path d="M475 270 L640 135" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="640,135 629,137 636,145" fill="#333"/>
</svg>

**J4 是 data/debug path，J12 是 main power path。两个路径都正确，target discovery 才有意义。**

## 4. 连接与上电

1. KV260 保持断电；
2. 把 USB data cable 从 development host 接到 **J4**；
3. 把 12 V / 3 A 电源接到 **J12**；
4. 上电；
5. 观察风扇/板上 power-status 指示是否符合官方 power-on 行为。

不要因为某个 LED 亮了就直接宣布“JTAG 正常”。Power evidence 与 target-discovery evidence 是两层。

## 5. 用课程脚本发现 target

从仓库根目录运行：

```bash
vivado -mode batch -nolog -nojournal \
  -source boards/kv260/scripts/detect_target.tcl \
  | tee lab-hw-02-target-detection.txt
```

脚本会：

1. 打开 Hardware Manager；
2. 连接本机 hw_server；
3. 打开可用 hardware target；
4. 打印 `get_hw_devices` 返回的 device；
5. 如果一个 device 都没有，返回非零状态。

**这一 Lab 不要求 device 里已经有我们的 bitstream。只要求 development host 真正看见 JTAG device。**

## 6. Expected Evidence

必须保存：

- `lab-hw-02-target-detection.txt`；
- board model + carrier revision；
- Vivado version；
- 脚本列出的 hardware device name(s)；
- Git commit；
- 日期。

通过标准不是“Hardware Manager 窗口打开了”，而是 **device list 非空并可重复发现**。建议断电重连一次后再次执行，确认不是偶然状态。

## 7. If it does not work

按顺序排查，**不要先改 RTL**：

1. **Power** — J12 是否真有 12 V？电源规格是否正确？
2. **Cable** — J4 是否接对？USB 是否支持 data？
3. **Driver** — LAB-HW-00 的 JTAG/cable driver 是否安装？
4. **Toolchain** — Vivado 版本与 board data 是否正确？
5. **Target** — 关闭其他可能独占 hw_server/JTAG 的 Vivado session，再重试。

如果脚本显示 `NO_HW_DEVICE`，这仍然是 bring-up failure，不是 neuron-design failure。

## 8. Human Check

1. 为什么 USB/J4 接上以后仍然必须给 J12 主供电？
2. “板子上电成功”和“JTAG target discovery 成功”各自证明什么？
3. 为什么今天不应该写任何 FlyBrain RTL？
4. 如果 `get_hw_devices` 为空，最先查哪三层？

## 9. 官方依据

AMD UG1089 Interfaces：  
https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Interfaces

AMD UG1089 Powering：  
https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Powering-the-Starter-Kit-and-Power-Budgets

AMD DS986 Power and Electrical：  
https://docs.amd.com/r/en-US/ds986-kv260-starter-kit/Power-and-Electrical